# Meaningful Latent Spaces: Using a semi-supervised autoencoder to infer position and ionization electrons from hit patterns

The idea was originally to have an autoencoder that let you have the cake and eat it too: having both better data compression and a(n) (partially) interpretable latent representation. The autoencoder architecture is built on a residual network that is composed of linear blocks and residual blocks (with 1 skip layer each). Originally written in 2023, the notebook has been since updated since the conference proceeding. In 2026, I cleaned up this notebook with LLM assistance (Claude), notably without the conference proceeding's cyclic annealing on the weights of the loss function components. 

The model simply reweighs the hit pattern reconstruction loss term now. This is the model only for simulated hit patterns (light sensor array measurements from simulations), but I have tested models on downsampled waveforms that perform reasonably well. 

Ultimately position reconstruction was better accomplished with conditional normalizing flows in XENONnT by my colleagues, energy reconstruction remains challenging due to the complexity of calibrating such a detector, and data compression would have had real impact on fully raw waveforms (PBs of data), but this project was a meaningful and interesting exercise to see what particle interaction event information it was possible to conserve while saving on storage space. 

This notebook is organized into the following sections:

1. **Setup:** imports, plotting style, GPU/threads, output directory
2. **Hyperparameters**
3. **Data loading:** read HDF5 files into a single array
4. **Preprocessing:** train/val/test split, dataloaders
5. **Model components: ** `LinBlock`, `ResBlockSkip1`, `AutoEncoder`
6. **Loss & optimizer:** Adam, LR scheduler on plateau
7. **Training loop**
8. **Save files:** save losses and model
9. **Evaluation:** check ne, position, and hit pattern reconstruction

## 1. Setup

In [ ]:
# Standard library
import os
import pickle
import random
from datetime import datetime

# Scientific stack
import numpy as np
import pandas as pd
import h5py
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

# PyTorch
import torch
from torch import nn, optim
from torch.utils.data import DataLoader
from torch.optim.lr_scheduler import StepLR

In [ ]:
# Plotting style (credit: Qin Juehang)
plt.style.use('default')
plt.rcParams.update({
    'font.family': 'serif',
    'figure.dpi': 300,
    'font.serif': 'cmr10',
    'font.size': 7.0,
    'axes.formatter.use_mathtext': True,
    'mathtext.fontset': 'cm',
})

cm = 1 / 2.54
single_column_figsize = (8 * cm, 6 * cm)
two_column_figsize = (16 * cm, 12 * cm)

In [ ]:
# GPU / threading
torch.set_num_threads(25)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch:", torch.__version__)
print("threads:", torch.get_num_threads())
print("device:", device)
if device.type == "cuda":
    print("gpu:", torch.cuda.get_device_name(0))
    print(f"allocated: {torch.cuda.memory_allocated(0) / 1024**3:.1f} GB")

%load_ext autoreload
%autoreload 2

## 2. Hyperparameters

In [ ]:
# Detector
N_PMT_TOT = 494
PMTList = list(map(str, range(N_PMT_TOT)))

# Architecture
SKIP = 1                      # number of layers skipped per ResBlock
ENCODER_DEPTH = 6             # number of ResBlocks in encoder
DECODER_DEPTH = 6             # number of ResBlocks in decoder
LSIZE = 10                    # latent space dimensions
LOSS_STRING = 'In-Out+Latent_MSE'  # for file naming

# Training
BATCH_SIZE = 500
STARTING_LR = 1e-4            
EPOCH_LR_DECAY = 5

# Data
PATHNAME = '/home/napoliion/posrec_dataset/posrec_dataset'
RUNID = '2000020'
SIGNAL = 'S2'
TEST_SIZE = 100000
RANDOM_SEED = 42

In [ ]:
# Output directory for this run
date = datetime.now().strftime("%B-%d-%y_%-I_%M_%-S_%p")
print("run date:", date)

OUTDIR = (
    f'ae_{date}_{LOSS_STRING}Loss_{SKIP}ResSkip_'
    f'{ENCODER_DEPTH}EncoderBlocks_{DECODER_DEPTH}DecoderBlocks_'
    f'{LSIZE}LatentDim_CHEP'
)

PATH = f'/home/napoliion/thesis_notebooks/{OUTDIR}'
os.makedirs(PATH, exist_ok=True)
print("output path:", PATH)

## 3. Data loading

In [ ]:
def load_hdf5_runs(pathname, signal, runid, filenums, keys):
    '''
    Load a list of HDF5 files for a given run/signal and return a list of
    dicts mapping each key to its loaded numpy array.
    '''
    df_list = []
    for filenum in filenums:
        num = str(filenum).zfill(4)
        file = f'{pathname}/{signal}_{runid}_{num}.hdf5'
        with h5py.File(file, 'r') as f:
            arr_dict = {key: np.array(f[key]) for key in keys}
        df_list.append(arr_dict)
    return df_list


def assemble_s2_matrix(df_list, n_pmt_tot, rows_per_file=5000):
    '''
    Pack the per-file dicts into a single (N, n_pmt_tot + 6) matrix:
        [0 : n_pmt_tot]      area_per_channel
        [n_pmt_tot]          n_electron
        [n_pmt_tot + 1]      npeaks
        [n_pmt_tot + 2 : +5] corr_pos (x, y, z)
        [n_pmt_tot + 5]      amplitude
    '''
    n_files = len(df_list)
    n_cols = n_pmt_tot + 6
    s2 = np.zeros((rows_per_file * n_files, n_cols))
    for i, d in enumerate(df_list):
        s, e = rows_per_file * i, rows_per_file * (i + 1)
        s2[s:e, 0:n_pmt_tot]                 = d['area_per_channel']
        s2[s:e, n_pmt_tot]                   = d['n_electron']
        s2[s:e, n_pmt_tot + 1:n_pmt_tot + 4] = d['corr_pos']
        s2[s:e, n_pmt_tot + 4]               = d['npeaks']
        s2[s:e, n_pmt_tot + 5]               = d['amplitude']
    return s2

In [ ]:
# Inspect available keys in one file (sanity check)
sample_path = os.path.join(PATHNAME, f'{SIGNAL}_{RUNID}_0103.hdf5')
with h5py.File(sample_path, 'r') as f:
    print("keys in", os.path.basename(sample_path), ":", list(f.keys()))

In [ ]:
# File 102 is excluded due to file transfer error in past, we have enough events to train so no need to go back.
KEYS = ['area', 'area_per_channel', 'n_electron', 'npeaks', 'corr_pos', 'amplitude']
filenums = list(range(0, 102)) + list(range(103, 200))

df_list = load_hdf5_runs(PATHNAME, SIGNAL, RUNID, filenums, KEYS)
s2_data = assemble_s2_matrix(df_list, N_PMT_TOT)
print("s2_data shape:", s2_data.shape)

## 4. Preprocessing

In [ ]:
def data_split(data, np_random_seed, test_size):
    '''
    Shuffle and split a 2-D numpy array into (train, val, test) where the
    test set has `test_size` rows and the remainder is split in half.
    '''
    np.random.seed(np_random_seed)
    indices = np.arange(len(data))
    np.random.shuffle(indices)
    data = data[indices]

    val_idx = data.shape[0] - test_size
    train_idx = int(val_idx / 2)

    return data[:train_idx].astype(np.float32), data[train_idx:val_idx].astype(np.float32), data[val_idx:].astype(np.float32)

In [ ]:
TPC_RADIUS = 66.4   # cm

def transform(raw_set, n_pmt_tot, tpc_radius):
    areas = raw_set[:, :n_pmt_tot]
    ne    = raw_set[:, n_pmt_tot]
    x     = raw_set[:, n_pmt_tot + 1]
    y     = raw_set[:, n_pmt_tot + 2]

    total = np.maximum(areas.sum(axis=1, keepdims=True), 1e-9)
    norm_pattern = areas / total
    log_total = np.log(total)
    encoder_input = np.concatenate([norm_pattern, log_total], axis=1)  # (N, 495)

    log_ne = np.log(ne + 1)
    x_norm = x / tpc_radius
    y_norm = y / tpc_radius
    targets = np.stack([log_ne, x_norm, y_norm], axis=1)               # (N, 3)

    return np.concatenate([encoder_input, targets], axis=1).astype(np.float32)  # (N, 498)

def inverse_targets(targets_norm, tpc_radius):
    log_ne, x_norm, y_norm = targets_norm[:, 0], targets_norm[:, 1], targets_norm[:, 2]
    ne = np.exp(log_ne) - 1
    x  = x_norm * tpc_radius
    y  = y_norm * tpc_radius
    return ne, x, y

In [ ]:
dataset = s2_data[:, 0:N_PMT_TOT + 1 + 3] # hit pattern, log(area), n_electrons, x, y

scaled_dataset = transform(dataset, N_PMT_TOT, TPC_RADIUS)

# Dataset solely for training: hitpatterns, n_electrons, x, y
train_set, val_set, test_set = data_split(
    scaled_dataset, np_random_seed=RANDOM_SEED, test_size=TEST_SIZE
)

# Full-feature copy (carries npeaks, corr_pos, amplitude, in case needed)
train_fullset, val_fullset, test_fullset = data_split(
    s2_data, np_random_seed=RANDOM_SEED, test_size=TEST_SIZE
)

print("train/val/test:", train_set.shape, val_set.shape, test_set.shape)
print("full     :    ", train_fullset.shape, val_fullset.shape, test_fullset.shape)

In [ ]:
def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

g = torch.Generator()
g.manual_seed(0)

train_loader = DataLoader(
    train_set,
    shuffle=True,
    batch_size=BATCH_SIZE,
    worker_init_fn=seed_worker,
    generator=g,
)
val_loader = DataLoader(val_set, batch_size=BATCH_SIZE)
test_loader = DataLoader(test_set, batch_size=TEST_SIZE)

## 5. Model components

`LinBlock` — single linear layer + optional activation.
`ResBlockSkip1` — two linear layers with a skip connection (input dim must equal output dim).
`AutoEncoder` — sequential stack of encoder modules, then sequential stack of decoder modules.


In [ ]:
class LinBlock(nn.Module):
    '''Single Linear layer with optional activation. No internal optimizer.'''

    def __init__(self, input_size, output_size, activation=None):
        super().__init__()
        self.input_size = input_size
        self.output_size = output_size
        self.lin1 = nn.Linear(input_size, output_size)
        self.activation = activation

    def forward(self, x):
        out = self.lin1(x)
        if self.activation is not None:
            out = self.activation(out)
        return out

In [ ]:
class ResBlockSkip1(nn.Module):
    '''
    Residual MLP block:  x -> Linear -> activation -> Linear -> (+x) -> activation
    input_size must equal output_size for the skip add to work.
    '''

    def __init__(self, input_size, output_size, activation=nn.ELU()):
        super().__init__()
        assert input_size == output_size, (
            f"ResBlockSkip1 requires input_size == output_size, got {input_size} vs {output_size}"
        )
        self.input_size = input_size
        self.output_size = output_size
        self.lin1 = nn.Linear(input_size, output_size)
        self.lin2 = nn.Linear(output_size, output_size)
        self.activation = activation

    def forward(self, x):
        residual = x
        out = self.activation(self.lin1(x))
        out = self.lin2(out)
        out = self.activation(out + residual)
        return out

In [ ]:
def build_module_list(layers):
    '''
    Build an nn.ModuleList from a list of (in, out, BlockClass, activation) tuples.
    '''
    return nn.ModuleList([
        block(in_size, out_size, activation)
        for in_size, out_size, block, activation in layers
    ])


class AutoEncoder(nn.Module):
    def __init__(self, encoder_layers, decoder_layers):
        super().__init__()
        self.encoder = build_module_list(encoder_layers)
        self.decoder = build_module_list(decoder_layers)

    def forward(self, inputs):
        x = inputs
        for layer in self.encoder:
            x = layer(x)
        encoded = x

        x = encoded
        for layer in self.decoder:
            x = layer(x)
        decoded = x

        return encoded, decoded


def get_params(model):
    '''Total number of trainable parameters.'''
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

### Build the model

The encoder is `ENCODER_DEPTH` ResBlocks at width `N_PMT_TOT`, then a final `LinBlock` that projects down to `LSIZE`. The decoder mirrors this structure with an extra projection at the front and an output projection at the end.


In [ ]:
encoder_layers = [(N_PMT_TOT + 1, N_PMT_TOT + 1, ResBlockSkip1, nn.ELU())] * ENCODER_DEPTH
encoder_layers += [(N_PMT_TOT + 1, LSIZE, LinBlock, nn.ELU())]

decoder_layers = [(LSIZE, N_PMT_TOT + 1, LinBlock, nn.ELU())]
decoder_layers += [(N_PMT_TOT + 1, N_PMT_TOT + 1, ResBlockSkip1, nn.ELU())] * DECODER_DEPTH
decoder_layers += [(N_PMT_TOT + 1, N_PMT_TOT + 1, LinBlock, nn.Softplus())]

model = AutoEncoder(encoder_layers, decoder_layers)
print(model)
print("trainable params:", get_params(model))

## 6. Loss and optimizer

In [ ]:
loss_function = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=STARTING_LR)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=3, threshold=1e-4
)

## 7. Training and validation

In [ ]:
def run_epoch(model, loader, loss_function, optimizer,
              n_pmt_tot, train,
              w_recon=1.0, w_ne=1.0, w_pos=1.0):
    sums = {'recon': 0.0, 'ne': 0.0, 'pos': 0.0, 'combined': 0.0}
    input_dim = n_pmt_tot + 1

    for batch in loader:
        X, y = torch.split(batch, [input_dim, 3], dim=1)
        X, y = X.float(), y.float()

        encoded, decoded = model(X)

        recon_loss = loss_function(decoded, X)
        ne_loss    = loss_function(encoded[:, 0], y[:, 0])
        # Euclidean distance loss on (x, y) — mean over batch of sqrt(dx^2 + dy^2)
        pos_loss   = torch.sqrt(((encoded[:, 1:3] - y[:, 1:3]) ** 2).sum(dim=1) + 1e-8).mean()

        loss = w_recon * recon_loss + w_ne * ne_loss + w_pos * pos_loss

        if train:
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        sums['recon']    += recon_loss.item()
        sums['ne']       += ne_loss.item()
        sums['pos']      += pos_loss.item()
        sums['combined'] += loss.item()

    return sums

In [ ]:
losses = {}
EPOCHS = 20

# Loss weights
W_RECON, W_NE, W_POS = 500.0, 1.0, 1.0

for epoch in range(EPOCHS):
    is_training_epoch = (epoch != 0)

    train_sums = run_epoch(
        model, train_loader, loss_function, optimizer, N_PMT_TOT,
        train=is_training_epoch,
        w_recon=W_RECON, w_ne=W_NE, w_pos=W_POS,
    )
    val_sums = run_epoch(
        model, val_loader, loss_function, optimizer, N_PMT_TOT,
        train=False,
        w_recon=W_RECON, w_ne=W_NE, w_pos=W_POS,
    )

    if is_training_epoch:
        scheduler.step(val_sums['combined'] / len(val_loader))

    n_train, n_val = len(train_loader), len(val_loader)
    print(
        f'epoch {epoch:3d}  '
        f'TRAIN: recon={train_sums["recon"]/n_train:.6f} '
        f'ne={train_sums["ne"]/n_train:.4f} '
        f'pos={train_sums["pos"]/n_train:.6f} || '
        f'VAL: recon={val_sums["recon"]/n_val:.4f} '
        f'ne={val_sums["ne"]/n_val:.4f} '
        f'pos={val_sums["pos"]/n_val:.6f} '
        f'lr={optimizer.param_groups[0]["lr"]:.2e}'
    )

    losses[epoch] = {'train': train_sums, 'val': val_sums,
                     'weights': (W_RECON, W_NE, W_POS)}

## 8. Save to file

In [ ]:
LOSSPATH = f'{PATH}/losses.pkl'
with open(LOSSPATH, 'wb') as f:
    pickle.dump(losses, f)

torch.save({
    'state_dict':    model.state_dict(),
    'optimizer':     optimizer.state_dict(),
    'scheduler':     scheduler.state_dict(),
    'config': {
        'N_PMT_TOT':     N_PMT_TOT,
        'LSIZE':         LSIZE,
        'ENCODER_DEPTH': ENCODER_DEPTH,
        'DECODER_DEPTH': DECODER_DEPTH,
        'TPC_RADIUS':    TPC_RADIUS,
    },
    'losses': losses,
}, f'{PATH}/model.pt')

print("saved losses to:", LOSSPATH)
print("saved model  to:", PATH)

## 9. Quick performance evaluations

In [ ]:
@torch.no_grad()
def collect_predictions(model, loader, n_pmt_tot, tpc_radius):
    '''Run the model over `loader` and return arrays in PHYSICAL units.'''
    model.eval()
    true_list, pred_list = [], []
    input_dim = n_pmt_tot + 1
    for batch in loader:
        X, y = torch.split(batch, [input_dim, 3], dim=1)
        X, y = X.float(), y.float()
        encoded, _ = model(X)
        true_list.append(y[:, 0:3].cpu().
                         numpy())
        pred_list.append(encoded[:, 0:3].cpu().numpy())
    model.train()
    true_norm = np.concatenate(true_list)
    pred_norm = np.concatenate(pred_list)

    # Invert to physical units
    true_ne = np.exp(true_norm[:, 0]) - 1
    pred_ne = np.exp(pred_norm[:, 0]) - 1
    true_x  = true_norm[:, 1] * tpc_radius
    pred_x  = pred_norm[:, 1] * tpc_radius
    true_y  = true_norm[:, 2] * tpc_radius
    pred_y  = pred_norm[:, 2] * tpc_radius

    return true_ne, pred_ne, true_x, pred_x, true_y, pred_y

true_ne, pred_ne, true_x, pred_x, true_y, pred_y = collect_predictions(
    model, test_loader, N_PMT_TOT, TPC_RADIUS
)
residual_ne = pred_ne - true_ne
residual_x  = pred_x  - true_x
residual_y  = pred_y  - true_y
pos_err     = np.sqrt(residual_x**2 + residual_y**2)
true_R      = np.sqrt(true_x**2 + true_y**2)

print(f'{"":>14}  {"mean":>10}  {"median":>10}  {"RMS":>10}')
print(f'{"n_e residual":>14}  {residual_ne.mean():>10.2f}  '
      f'{np.median(residual_ne):>10.2f}  {np.sqrt((residual_ne**2).mean()):>10.2f}  electrons')
print(f'{"|Δr|":>14}  {pos_err.mean():>10.3f}  '
      f'{np.median(pos_err):>10.3f}  {np.sqrt((pos_err**2).mean()):>10.3f}  cm')

In [ ]:
rng = np.random.default_rng(0)
n_plot = min(20000, len(true_ne))
idx = rng.choice(len(true_ne), size=n_plot, replace=False)

# Percentage difference for n_e: 100 * (pred - true) / true
pct_residual_ne = 100.0 * residual_ne / np.maximum(true_ne, 1.0)

fig, axes = plt.subplots(2, 2, figsize=(two_column_figsize[0], two_column_figsize[1] * 1.4))

# --- n_e residual vs true (raw) ---
ax = axes[0, 0]
ax.scatter(true_ne[idx], residual_ne[idx], s=1, alpha=0.2, rasterized=True)
ax.axhline(0, color='k', lw=0.6)
ax.set_xlabel(r'true $n_e$')
ax.set_ylabel(r'pred $-$ true')

# --- n_e percent residual vs true (raw) ---
ax = axes[0, 1]
ax.scatter(true_ne[idx], pct_residual_ne[idx], s=1, alpha=0.2, rasterized=True)
ax.axhline(0, color='k', lw=0.6)
ax.set_xlabel(r'true $n_e$')
ax.set_ylabel(r'$100 \times (\mathrm{pred} - \mathrm{true})/\mathrm{true}$ [%]')

# --- position error magnitude vs R ---
ax = axes[1, 0]
ax.scatter(true_R[idx], pos_err[idx], s=1, alpha=0.2, rasterized=True)
ax.axhline(0, color='k', lw=0.6)
ax.set_xlabel(r'true $R$ [cm]')
ax.set_ylabel(r'$|\Delta \vec{r}|$ [cm]')

# --- 2D residual scatter ---
ax = axes[1, 1]
ax.scatter(residual_x[idx], residual_y[idx], s=1, alpha=0.2, rasterized=True)
ax.axhline(0, color='k', lw=0.4)
ax.axvline(0, color='k', lw=0.4)
ax.set_aspect('equal')
ax.set_xlabel(r'$\Delta x$ [cm]')
ax.set_ylabel(r'$\Delta y$ [cm]')

fig.tight_layout()
plt.show()

In [ ]:
n_bins = 10
edges = np.quantile(true_R, np.linspace(0, 1, n_bins + 1))
edges = np.unique(edges)

centers, med_err, p84_err, counts = [], [], [], []
for b in range(len(edges) - 1):
    mask = (true_R >= edges[b]) & (true_R < edges[b + 1])
    if mask.sum() < 50: continue
    centers.append(true_R[mask].mean())
    med_err.append(np.median(pos_err[mask]))
    p84_err.append(np.quantile(pos_err[mask], 0.84))
    counts.append(mask.sum())

print(f'{"R [cm]":>10}  {"median |Δr|":>14}  {"P84 |Δr|":>12}  {"N":>8}')
for c, m, p, n in zip(centers, med_err, p84_err, counts):
    print(f'{c:>10.1f}  {m:>14.2f}  {p:>12.2f}  {n:>8d}')

In [ ]:
@torch.no_grad()
def collect_reconstructions(model, loader, n_pmt_tot, k_keep=200):
    '''Return (true_X, decoded_X) for the first k_keep test events.'''
    model.eval()
    true_chunks, dec_chunks = [], []
    seen = 0
    input_dim = n_pmt_tot + 1
    for batch in loader:
        X, _y = torch.split(batch, [input_dim, 3], dim=1)
        X = X.float()
        _, decoded = model(X)
        true_chunks.append(X.cpu().numpy())
        dec_chunks.append(decoded.cpu().numpy())
        seen += X.shape[0]
        if seen >= k_keep:
            break
    model.train()
    true_X = np.concatenate(true_chunks)[:k_keep]
    dec_X  = np.concatenate(dec_chunks)[:k_keep]
    return true_X, dec_X

true_X, dec_X = collect_reconstructions(model, test_loader, N_PMT_TOT, k_keep=200)
true_pattern, true_total = true_X[:, :N_PMT_TOT], true_X[:, N_PMT_TOT]
dec_pattern,  dec_total  = dec_X[:, :N_PMT_TOT],  dec_X[:, N_PMT_TOT]

# Per-event reconstruction MSE on the pattern (the 494-channel part)
per_event_mse = ((dec_pattern - true_pattern) ** 2).mean(axis=1)
print(f'pattern MSE   median: {np.median(per_event_mse):.2e}')
print(f'pattern MSE   P95   : {np.quantile(per_event_mse, 0.95):.2e}')

# Total-area comparison: model's predicted total vs. true total, and
# vs. the sum of the decoded pattern itself (these two should agree if training is healthy)
print(f'total residual (dec_total - true_total)        mean: {(dec_total - true_total).mean():.3f}')
print(f'total residual (dec_total - true_total)        std : {(dec_total - true_total).std():.3f}')
print(f'sum-of-decoded-pattern vs true_total residual  mean: {(dec_pattern.sum(axis=1) - true_total).mean():.3f}')
print(f'sum-of-decoded-pattern vs true_total residual  std : {(dec_pattern.sum(axis=1) - true_total).std():.3f}')

In [ ]:
pmt_pos_df = pd.read_csv('/home/napoliion/energy-reconstruction/straxen-positions.csv')
pmt_pos_df = pmt_pos_df.sort_values('i').reset_index(drop=True)
pmt_pos_df = pmt_pos_df.iloc[:N_PMT_TOT].copy()
top_mask = pmt_pos_df['array'].str.lower().str.strip().eq('top').values

pmt_xy_top = pmt_pos_df.loc[top_mask, ['x', 'y']].values
top_channels = np.where(top_mask)[0]   # <-- defined here

In [ ]:
# Pick a mix: best, median, worst reconstructions
order = np.argsort(per_event_mse)
picks = {
    'Best':   order[:3],
    'Middle': order[len(order)//2 : len(order)//2 + 3],
    'Worst':  order[-3:],
}
n_show = 9
true_pattern_raw = true_pattern
dec_pattern_raw  = dec_pattern

# Percentage difference per PMT: 100 * (dec - true) / true
# Floor to avoid blowups on near-zero PMTs; clip display to ±100% so the colormap
# isn't dominated by a single huge outlier on a dim PMT.
EPS = 1e-3   # set to roughly your detection-threshold area-per-channel
pct_diff = 100.0 * (dec_pattern_raw - true_pattern_raw) / np.maximum(np.abs(true_pattern_raw), EPS)

fig, axes = plt.subplots(3, n_show,
                         figsize=(two_column_figsize[0] * 1.2, 3),
                         gridspec_kw={'hspace': 0.05, 'wspace': 0.05})

# Log color scale for the true/decoded rows
shown = np.concatenate(list(picks.values()))
all_vals = np.concatenate([true_pattern_raw[shown][:, top_channels],
                           dec_pattern_raw[shown][:, top_channels]])
vmax = all_vals.max()
vmin = max(all_vals[all_vals > 0].min(), 1e-3)
norm = mcolors.LogNorm(vmin=vmin, vmax=vmax)

# Symmetric diverging scale for percentage difference, clipped to ±100%
PCT_LIM = 100.0
pct_norm = mcolors.Normalize(vmin=-PCT_LIM, vmax=PCT_LIM)

j = 0
for label, indices in picks.items():
    for idx in indices:
        # Row 0: true
        ax = axes[0, j]
        apc = np.clip(true_pattern_raw[idx, top_channels], vmin, None)
        ax.scatter(pmt_xy_top[:, 0], pmt_xy_top[:, 1], c=apc, s=1, norm=norm, cmap='viridis')
        ax.set_aspect('equal'); ax.set_xticks([]); ax.set_yticks([])
        ax.set_title(f'{label}\nMSE={per_event_mse[idx]:.1e}', fontsize=6)

        # Row 1: decoded
        ax = axes[1, j]
        apc = np.clip(dec_pattern_raw[idx, top_channels], vmin, None)
        sc = ax.scatter(pmt_xy_top[:, 0], pmt_xy_top[:, 1], c=apc, s=1, norm=norm, cmap='viridis')
        ax.set_aspect('equal'); ax.set_xticks([]); ax.set_yticks([])

        # Row 2: percentage difference (clipped for display)
        ax = axes[2, j]
        pct = np.clip(pct_diff[idx, top_channels], -PCT_LIM, PCT_LIM)
        sc_pct = ax.scatter(pmt_xy_top[:, 0], pmt_xy_top[:, 1], c=pct, s=1,
                            norm=pct_norm, cmap='RdBu_r')
        ax.set_aspect('equal'); ax.set_xticks([]); ax.set_yticks([])
        j += 1

axes[0, 0].set_ylabel('Original', fontsize=7)
axes[1, 0].set_ylabel('Decoded', fontsize=7)
axes[2, 0].set_ylabel('% diff', fontsize=7)

cbar = fig.colorbar(sc, ax=axes[0:2, :].ravel().tolist(), shrink=0.9, pad=0.02)
cbar.set_label('Area per channel [PE]')

cbar_pct = fig.colorbar(sc_pct, ax=axes[2, :].ravel().tolist(), shrink=0.9, pad=0.02)
cbar_pct.set_label(r'$\%$ diff $[\%]$')

plt.show()